This notebook creates a SQLite database using an xlsx file

In [8]:
!pip install pandas openpyxl


[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import pandas as pd

file = "dbmsdata.xlsx"

player_df = pd.read_excel(file, sheet_name="Player", header=1)
team_df = pd.read_excel(file, sheet_name="Team", header = 1)
game_df = pd.read_excel(file, sheet_name="Game", header = 1)
performance_df = pd.read_excel(file, sheet_name="Performance", header = 1)
qb_df = pd.read_excel(file, sheet_name="QuarterbackStats", header = 1)
wr_df = pd.read_excel(file, sheet_name="WideReceiverStats", header = 1)
injury_df = pd.read_excel(file, sheet_name="Injury", header = 1)
position_df = pd.read_excel(file, sheet_name="Position", header=1)
school_df = pd.read_excel(file, sheet_name="School", header=1)
sport_df = pd.read_excel(file, sheet_name="Sport", header=1)
bb_df = pd.read_excel(file, sheet_name="BasketBallStats", header=1)

In [10]:
import sqlite3

conn = sqlite3.connect("sports.db")

In [11]:
player_df.to_sql("Player", conn, if_exists="replace", index=False)
team_df.to_sql("Team", conn, if_exists="replace", index=False)
game_df.to_sql("Game", conn, if_exists="replace", index=False)
performance_df.to_sql("Performance", conn, if_exists="replace", index=False)
qb_df.to_sql("QuarterbackStats", conn, if_exists="replace", index=False)
wr_df.to_sql("WideReceiverStats", conn, if_exists="replace", index=False)
injury_df.to_sql("Injury", conn, if_exists="replace", index=False)
position_df.to_sql("Position", conn, if_exists="replace", index=False)
school_df.to_sql("School", conn, if_exists="replace", index=False)
sport_df.to_sql("Sport", conn, if_exists="replace", index=False)
bb_df.to_sql("BasketBallStats", conn, if_exists="replace", index=False)

8

In [12]:
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)

,name
0,Player
1,Team
2,Game
3,Performance
4,QuarterbackStats
5,WideReceiverStats
6,Injury
7,Position
8,School
9,Sport


In [13]:
query = """
SELECT Player.FirstName, Player.LastName, Performance.RANKING
FROM Player
JOIN Performance 
ON Player.PlayerID = Performance.PlayerID;
"""

pd.read_sql(query, conn)

,FirstName,LastName,RANKING
0,Aaron,Anderson,1
1,Aaron,Anderson,2
2,Aaron,Anderson,2
3,Aaron,Anderson,2
4,Aaron,Anderson,3
5,Nic,Anderson,1
6,Nic,Anderson,1
7,Nic,Anderson,2
8,Nic,Anderson,3
9,Zavion,Thomas,2


In [14]:
query1 = """
SELECT Player.FirstName, Player.LastName, 
       Performance.GameID, Performance.RANKING
FROM Player
JOIN Performance 
ON Player.PlayerID = Performance.PlayerID
ORDER BY Performance.GameID, Performance.RANKING;
"""

pd.read_sql(query1, conn)

,FirstName,LastName,GameID,RANKING
0,Garrett,Nussmeier,G001,1
1,Aaron,Anderson,G001,2
2,Nic,Anderson,G001,3
3,Germie,Bernard,G001,4
4,Isaiah,Horton,G001,5
5,Garrett,Nussmeier,G002,1
6,Aaron,Anderson,G002,2
7,Zavion,Thomas,G002,3
8,Germie,Bernard,G002,4
9,Jaylen,Mbakwe,G002,5


In [15]:
query1 = """
SELECT Player.FirstName, Player.LastName, 
       Performance.GameID, Performance.RANKING
FROM Player
JOIN Performance 
ON Player.PlayerID = Performance.PlayerID
WHERE Performance.Ranking = 1
ORDER BY Performance.GameID;
"""

pd.read_sql(query1, conn)

,FirstName,LastName,GameID,RANKING
0,Garrett,Nussmeier,G001,1
1,Garrett,Nussmeier,G002,1
2,Nic,Anderson,G003,1
3,Garrett,Nussmeier,G004,1
4,Nic,Anderson,G005,1
5,Aaron,Anderson,G006,1
6,Garrett,Nussmeier,G007,1
7,CJ,Wiley,G008,1
8,Jessica,Timmons,G010,1
9,Jessica,Timmons,G012,1


In [16]:
query2 = """
SELECT Team.TeamID, Team.Budget,
       COUNT(*) AS TopPerformers
FROM Team
JOIN Position ON Team.TeamID = Position.TeamID
JOIN Performance ON Position.PlayerID = Performance.PlayerID
WHERE Performance.RANKING = 1
GROUP BY Team.TeamID
ORDER BY TopPerformers DESC;
"""

pd.read_sql(query2, conn)

,TeamID,Budget,TopPerformers
0,T001,40000000,14
1,T013,40000000,6
2,T004,25000000,2


In [17]:
query3 = """
SELECT Team.TeamID, Team.Budget,
       COUNT(DISTINCT Injury.PlayerID) AS InjuredPlayers
FROM Team
JOIN Position ON Team.TeamID = Position.TeamID
JOIN Injury ON Position.PlayerID = Injury.PlayerID
GROUP BY Team.TeamID
ORDER BY InjuredPlayers DESC;
"""

pd.read_sql(query3, conn)

,TeamID,Budget,InjuredPlayers
0,T001,40000000,5
1,T004,25000000,2
2,T002,20000000,2


In [18]:
query4 = """
SELECT H.TeamID, H.HomeWins, A.AwayWins
FROM 
    (SELECT Team.TeamID, COUNT(*) AS HomeWins
     FROM Team
     JOIN Game ON Team.TeamID = Game.HomeTID
     WHERE Game.HomeScore > Game.AwayScore
     GROUP BY Team.TeamID) AS H

JOIN

    (SELECT Team.TeamID, COUNT(*) AS AwayWins
     FROM Team
     JOIN Game ON Team.TeamID = Game.AwayTID
     WHERE Game.AwayScore > Game.HomeScore
     GROUP BY Team.TeamID) AS A

ON H.TeamID = A.TeamID
ORDER BY H.TeamID;
"""

pd.read_sql(query4, conn)

,TeamID,HomeWins,AwayWins
0,T001,3,3


In [19]:
query5 = """
SELECT Team.TeamID, 'Home' AS Location, COUNT(*) AS Wins
FROM Team
JOIN Game ON Team.TeamID = Game.HomeTID
WHERE Game.HomeScore > Game.AwayScore
GROUP BY Team.TeamID

UNION

SELECT Team.TeamID, 'Away' AS Location, COUNT(*) AS Wins
FROM Team
JOIN Game ON Team.TeamID = Game.AwayTID
WHERE Game.AwayScore > Game.HomeScore
GROUP BY Team.TeamID

ORDER BY TeamID, Location;
"""

pd.read_sql(query5, conn)

,TeamID,Location,Wins
0,T001,Away,3
1,T001,Home,3
2,T006,Away,1
3,T009,Home,1
4,T011,Home,2
5,T012,Away,1
6,T014,Home,2


In [20]:
query6 = """
SELECT School.SchoolName, Team.TeamID, 'Home' AS Location, COUNT(*) AS Wins
FROM Team
JOIN School ON Team.SchoolID = School.SchoolID
JOIN Game ON Team.TeamID = Game.HomeTID
WHERE Game.HomeScore > Game.AwayScore
GROUP BY Team.TeamID

UNION

SELECT School.SchoolName, Team.TeamID, 'Away' AS Location, COUNT(*) AS Wins
FROM Team
JOIN School ON Team.SchoolID = School.SchoolID
JOIN Game ON Team.TeamID = Game.AwayTID
WHERE Game.AwayScore > Game.HomeScore
GROUP BY Team.TeamID

ORDER BY TeamID, Location;
"""

pd.read_sql(query6, conn)

,SchoolName,TeamID,Location,Wins
0,LSU,T001,Away,3
1,LSU,T001,Home,3
2,Florida,T006,Away,1
3,Clemson,T009,Home,1
4,LSU,T011,Home,2
5,Florida,T012,Away,1
6,Texas A&M,T014,Home,2


In [21]:
query7 = """
SELECT Player.FirstName, Player.LastName,
       School.SchoolName,
       Sport.SportName,
       Position.Position,
       Player.Year,
       Performance.RANKING
FROM Player
JOIN Position ON Player.PlayerID = Position.PlayerID
JOIN Team ON Position.TeamID = Team.TeamID
JOIN School ON Team.SchoolID = School.SchoolID
JOIN Sport ON Team.SportID = Sport.SportID
JOIN Performance ON Player.PlayerID = Performance.PlayerID
WHERE Performance.GameID = (
    SELECT MAX(GameID)
    FROM Performance AS P2
    WHERE P2.PlayerID = Player.PlayerID
);
"""

pd.read_sql(query7, conn)

,FirstName,LastName,SchoolName,SportName,Position,Year,RANKING
0,Aaron,Anderson,LSU,Football,WR,Junior,3
1,Aaron,Anderson,LSU,Football,WR,Junior,3
2,Nic,Anderson,LSU,Football,WR,Junior,2
3,Nic,Anderson,LSU,Football,WR,Junior,2
4,Zavion,Thomas,LSU,Football,WR,Senior,2
5,Zavion,Thomas,LSU,Football,WR,Senior,2
6,Germie,Bernard,Alabama,Football,WR,Senior,3
7,Germie,Bernard,Alabama,Football,WR,Senior,3
8,Isaiah,Horton,Alabama,Football,WR,Junior,3
9,Isaiah,Horton,Alabama,Football,WR,Junior,3


In [22]:
query8 = """
SELECT Player.FirstName, Player.LastName,
       Position.Position,
       Position.Number,
       Player.Year
FROM Player
JOIN Position ON Player.PlayerID = Position.PlayerID
WHERE Position.TeamID = 'T001'
ORDER BY Position.Position, Position.Number;
"""

pd.read_sql(query8, conn)

,FirstName,LastName,Position,Number,Year
0,Garrett,Nussmeier,QB,13,Senior
1,Garrett,Nussmeier,QB,13,Senior
2,Michael,Van Buren,QB,15,Sophomore
3,Michael,Van Buren,QB,15,Sophomore
4,Zavion,Thomas,WR,0,Senior
5,Zavion,Thomas,WR,0,Senior
6,Aaron,Anderson,WR,1,Junior
7,Aaron,Anderson,WR,1,Junior
8,Nic,Anderson,WR,4,Junior
9,Nic,Anderson,WR,4,Junior


In [23]:
query9 = """
SELECT GameID, Date, HomeTID, AwayTID, HomeScore, AwayScore
FROM Game
WHERE (HomeTID = 'T001' AND AwayTID = 'T002')
   OR (HomeTID = 'T002' AND AwayTID = 'T001');
"""

pd.read_sql(query9, conn)

,GameID,Date,HomeTID,AwayTID,HomeScore,AwayScore
0,G001,2025-09-06 00:00:00,T001,T002,42,17


In [24]:
query10 = """
SELECT Player.FirstName, Player.LastName,
       WideReceiverStats.GameID,
       WideReceiverStats.ReceivingYds
FROM WideReceiverStats
JOIN Player ON WideReceiverStats.PlayerID = Player.PlayerID
WHERE WideReceiverStats.ReceivingYds >= 100
ORDER BY WideReceiverStats.ReceivingYds DESC;
"""

pd.read_sql(query10, conn)

,FirstName,LastName,GameID,ReceivingYds
0,Nic,Anderson,G003,134
1,Nic,Anderson,G005,121
2,Nic,Anderson,G007,118
3,Aaron,Anderson,G001,112
4,Aaron,Anderson,G006,110
5,Aaron,Anderson,G004,105


In [25]:
query11 = """
SELECT Player.FirstName, Player.LastName,
       SUM(WideReceiverStats.ReceivingYds) AS TotalYards
FROM WideReceiverStats
JOIN Player ON WideReceiverStats.PlayerID = Player.PlayerID
GROUP BY Player.PlayerID
HAVING TotalYards >= 100
ORDER BY TotalYards DESC;
"""

pd.read_sql(query11, conn)

,FirstName,LastName,TotalYards
0,Aaron,Anderson,492
1,Nic,Anderson,457
2,Zavion,Thomas,133


In [26]:
query12 = """
SELECT Position.Season,
       Player.FirstName,
       Player.LastName,
       AVG(QuarterbackStats.PassingYds) AS AvgPassingYards
FROM QuarterbackStats
JOIN Player ON QuarterbackStats.PlayerID = Player.PlayerID
JOIN Position ON Player.PlayerID = Position.PlayerID
GROUP BY Position.Season, Player.PlayerID
ORDER BY Position.Season, AvgPassingYards DESC;
"""

pd.read_sql(query12, conn)

,Season,FirstName,LastName,AvgPassingYards
0,2025,Garrett,Nussmeier,278.571429
1,2025,Michael,Van Buren,210.000000
2,2026,Garrett,Nussmeier,278.571429
3,2026,Michael,Van Buren,210.000000


In [27]:
cursor = conn.cursor()

cursor.execute("""
UPDATE Injury
SET EndDate = NULL
WHERE PlayerID = 'P011';
""")
conn.commit()

In [28]:
cursor = conn.cursor()

cursor.execute("""
UPDATE Injury
SET EndDate = '2025-11-15'
WHERE PlayerID = 'P001' AND Type = 'Hamstring' AND StartDate = '2025-10-20';

""")
conn.commit()

In [29]:
query_current_injuries = """
SELECT Player.FirstName, Player.LastName,
       Injury.Type,
       Injury.StartDate,
       Injury.EndDate
FROM Injury
JOIN Player ON Injury.PlayerID = Player.PlayerID
WHERE Injury.EndDate IS NULL;
"""

pd.read_sql(query_current_injuries, conn)

,FirstName,LastName,Type,StartDate,EndDate
0,Michael,Van Buren,Wrist Soreness,2025-11-03 00:00:00,None
1,Aaron,Anderson,Ankle Sprain,2026-09-21 00:00:00,None
2,Nic,Anderson,Hamstring Strain,2026-09-28 00:00:00,None
3,Germie,Bernard,Shoulder Bruise,2026-10-04 00:00:00,None
4,London,Humphreys,Wrist Soreness,2026-11-03 00:00:00,None
5,Nyceara,Pryor,Concussion,2026-11-03 00:00:00,None


In [30]:
query_wins_season = """
SELECT TeamID, Season, SUM(Wins) AS TotalWins
FROM (
    SELECT HomeTID AS TeamID,
           substr(Date, 1, 4) AS Season,
           COUNT(*) AS Wins
    FROM Game
    WHERE HomeScore > AwayScore
    GROUP BY HomeTID, substr(Date, 1, 4)

    UNION ALL

    SELECT AwayTID AS TeamID,
           substr(Date, 1, 4) AS Season,
           COUNT(*) AS Wins
    FROM Game
    WHERE AwayScore > HomeScore
    GROUP BY AwayTID, substr(Date, 1, 4)
) AS TeamWins
GROUP BY TeamID, Season
ORDER BY Season, TotalWins DESC;
"""

pd.read_sql(query_wins_season, conn)

,TeamID,Season,TotalWins
0,T001,2025,6
1,T006,2025,1
2,T009,2025,1
3,T011,2026,2
4,T014,2026,2
5,T012,2026,1


In [31]:
query_school_teams_budget = """
SELECT School.SchoolName,
       Team.TeamID,
       Team.Budget,
       (
         SELECT COUNT(*)
         FROM Position
         WHERE Position.TeamID = Team.TeamID
       ) AS NumberOfPlayers
FROM School
JOIN Team ON School.SchoolID = Team.SchoolID
WHERE School.SchoolID = 'S001'
ORDER BY Team.Budget DESC;
"""

pd.read_sql(query_school_teams_budget, conn)

,SchoolName,TeamID,Budget,NumberOfPlayers
0,LSU,T011,50000000,2
1,LSU,T001,40000000,10


In [32]:
query_not_injured_not_top = """
SELECT Player.PlayerID, Player.FirstName, Player.LastName
FROM Player
WHERE NOT EXISTS (
    SELECT *
    FROM Injury
    WHERE Injury.PlayerID = Player.PlayerID
)
AND NOT EXISTS (
    SELECT *
    FROM Performance
    WHERE Performance.PlayerID = Player.PlayerID
      AND Performance.RANKING = 1
);
"""

pd.read_sql(query_not_injured_not_top, conn)

,PlayerID,FirstName,LastName
0,P006,Jaylen,Mbakwe
1,P012,Flau'jae,Johnson


In [33]:
query15 = """
SELECT Player.PlayerID,
       Player.FirstName,
       Player.LastName,
       School.SchoolName,
       Team.TeamID,
       Position.Position,
       Position.Season
FROM Player
JOIN Position ON Player.PlayerID = Position.PlayerID
JOIN Team ON Position.TeamID = Team.TeamID
JOIN School ON Team.SchoolID = School.SchoolID
WHERE NOT EXISTS (
    SELECT *
    FROM Injury
    WHERE Injury.PlayerID = Player.PlayerID
)
AND NOT EXISTS (
    SELECT *
    FROM Performance
    WHERE Performance.PlayerID = Player.PlayerID
      AND Performance.RANKING = 1
)
AND EXISTS (
    SELECT *
    FROM Performance
    WHERE Performance.PlayerID = Player.PlayerID
)
ORDER BY School.SchoolName, Team.TeamID, Player.PlayerID;
"""

pd.read_sql(query15, conn)

,PlayerID,FirstName,LastName,SchoolName,TeamID,Position,Season
0,P006,Jaylen,Mbakwe,Alabama,T002,WR,2025
1,P006,Jaylen,Mbakwe,Alabama,T002,WR,2026
2,P012,Flau'jae,Johnson,LSU,T011,G,2025
3,P012,Flau'jae,Johnson,LSU,T011,G,2026


In [34]:
query16 = """
SELECT School.SchoolName,
       Team.TeamID,
       Team.Budget
FROM Team
JOIN School ON Team.SchoolID = School.SchoolID
WHERE Team.Budget > (
    SELECT AVG(Budget)
    FROM Team
)
ORDER BY Team.Budget DESC;
"""

pd.read_sql(query16, conn)

,SchoolName,TeamID,Budget
0,LSU,T011,50000000
1,LSU,T001,40000000
2,Alabama,T013,40000000
3,Texas A&M,T005,30000000
4,Florida,T012,30000000
